In [ ]:
# ======================
# 0. SETUP & IMPORTS
# ======================
import os
import time
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from os import listdir
from sklearn.preprocessing import LabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
from keras.models import Model, load_model
from keras.layers import Dense, Dropout, GlobalAveragePooling2D, BatchNormalization
from keras.optimizers import Adam
from keras.callbacks import ReduceLROnPlateau, EarlyStopping, CSVLogger
from tensorflow.keras.preprocessing.image import img_to_array, ImageDataGenerator
from keras.applications import VGG16, ResNet50, MobileNetV2, InceptionV3, Xception, EfficientNetB0
from google.colab import drive

# Mount Google Drive
# drive.mount('/content/drive')

# Create output directory
experiment_name = f"bird_species_{time.strftime('%Y%m%d_%H%M%S')}"
output_dir = f"/content/drive/MyDrive/SEM 6/ML/CODE/COMPARISON/{experiment_name}"
os.makedirs(output_dir, exist_ok=True)

# ======================
# 1. DATA PREPARATION
# ======================
def load_data(limit_per_class=200):
    """Load and preprocess images"""
    print("[1/6] Loading and preprocessing data...")
    dataset_path = "/content/drive/MyDrive/SEM 6/ML/CODE/Bird Speciees Dataset"
    images, labels = [], []

    for class_folder in sorted(listdir(dataset_path)):
        class_path = os.path.join(dataset_path, class_folder)
        for img_file in listdir(class_path)[:limit_per_class]:
            img = cv2.imread(os.path.join(class_path, img_file))
            if img is not None:
                img = cv2.resize(img, (224, 224))
                images.append(img_to_array(img))
                labels.append(class_folder)

    # Convert to numpy arrays
    images = np.array(images, dtype="float32") / 255.0
    lb = LabelBinarizer()
    labels = lb.fit_transform(labels)

    # Save class names
    with open(f"{output_dir}/class_names.txt", "w") as f:
        f.write("\n".join(lb.classes_))

    return images, labels, lb

# Load and split data
images, labels, label_binarizer = load_data()
X_train, X_test, y_train, y_test = train_test_split(images, labels, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.15, random_state=42)  # 68%/17%/15% split

# Save dataset statistics
dataset_stats = {
    "total_samples": len(images),
    "train_samples": len(X_train),
    "val_samples": len(X_val),
    "test_samples": len(X_test),
    "image_shape": X_train[0].shape
}
pd.DataFrame.from_dict(dataset_stats, orient='index').to_csv(f"{output_dir}/dataset_stats.csv")

# ======================
# 2. DATA AUGMENTATION
# ======================
print("[2/6] Setting up data augmentation...")
train_datagen = ImageDataGenerator(
    rotation_range=25,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.15,
    zoom_range=0.15,
    horizontal_flip=True,
    fill_mode="nearest"
)
train_generator = train_datagen.flow(X_train, y_train, batch_size=32)

# ======================
# 3. MODEL DEFINITION
# ======================
def build_transfer_model(base_model_fn, fine_tune_layers=0):
    """Build model with controlled fine-tuning"""
    print(f"[3/6] Building model with {fine_tune_layers} fine-tuned layers...")

    # Load base model
    base_model = base_model_fn(
        weights="imagenet",
        include_top=False,
        input_shape=(224, 224, 3)
    )

    # Freezing strategy
    base_model.trainable = False  # Freeze all by default

    if fine_tune_layers > 0:
        base_model.trainable = True
        # Freeze all except last N layers
        for layer in base_model.layers[:-fine_tune_layers]:
            layer.trainable = False

    # Add custom head
    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(512, activation='relu')(x)
    x = Dropout(0.5)(x)
    x = BatchNormalization()(x)
    predictions = Dense(len(label_binarizer.classes_), activation='softmax')(x)

    model = Model(inputs=base_model.input, outputs=predictions)

    # Learning rate strategy
    lr = 1e-5 if fine_tune_layers > 0 else 1e-4

    model.compile(
        optimizer=Adam(learning_rate=lr),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

# Model configurations
MODELS = {
    "VGG16": (VGG16, 5),        # Fine-tune last 5/23 layers
    "ResNet50": (ResNet50, 10),  # Fine-tune last 10/175 layers
    "MobileNetV2": (MobileNetV2, 0),  # No fine-tuning
    "Xception": (Xception, 15),  # Fine-tune last 15/132 layers
    "EfficientNetB0": (EfficientNetB0, 8)
}

# ======================
# 4. MODEL TRAINING
# ======================
results = []

for model_name, (model_fn, fine_tune_layers) in MODELS.items():
    print(f"\n[4/6] Training {model_name}...")
    model_dir = f"{output_dir}/{model_name}"
    os.makedirs(model_dir, exist_ok=True)

    # Build model
    model = build_transfer_model(model_fn, fine_tune_layers)

    # Callbacks
    callbacks = [
        EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3),
        CSVLogger(f"{model_dir}/training_log.csv")
    ]

    # Train
    start_time = time.time()
    history = model.fit(
        train_generator,
        epochs=30,
        steps_per_epoch=len(X_train) // 32,
        validation_data=(X_val, y_val),
        callbacks=callbacks,
        verbose=1
    )
    training_time = time.time() - start_time

    # Get best epoch from history (accounting for early stopping)
    best_epoch = len(history.history['val_loss']) - callbacks[0].patience - 1

    # Get training metrics at best epoch
    train_acc = history.history['accuracy'][best_epoch]
    train_loss = history.history['loss'][best_epoch]
    val_acc = history.history['val_accuracy'][best_epoch]
    val_loss = history.history['val_loss'][best_epoch]

    # Evaluate on test set
    test_start = time.time()
    test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
    inference_time = (time.time() - test_start) / len(X_test)  # Per sample

    # Calculate parameters
    trainable_params = sum([np.prod(w.shape) for w in model.trainable_weights])
    non_trainable_params = sum([np.prod(w.shape) for w in model.non_trainable_weights])
    total_params = model.count_params()

    # Save model
    model.save(f"{model_dir}/model.h5")

    # Generate predictions
    y_pred = model.predict(X_test)
    y_pred_classes = np.argmax(y_pred, axis=1)
    y_true_classes = np.argmax(y_test, axis=1)

    # Save classification report
    report = classification_report(
        y_true_classes, y_pred_classes,
        target_names=label_binarizer.classes_,
        output_dict=True
    )
    pd.DataFrame(report).transpose().to_csv(f"{model_dir}/classification_report.csv")

    # Save confusion matrix
    plt.figure(figsize=(10, 8))
    cm = confusion_matrix(y_true_classes, y_pred_classes)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=label_binarizer.classes_,
                yticklabels=label_binarizer.classes_)
    plt.title(f"{model_name} Confusion Matrix")
    plt.savefig(f"{model_dir}/confusion_matrix.png")
    plt.close()

    # Store all required results
    model_results = {
        'Model': model_name,
        'Trainable_Params': trainable_params,
        'Non_Trainable_Params': non_trainable_params,
        'Total_Params': total_params,
        'Train_Samples': len(X_train),
        'Val_Samples': len(X_val),
        'Test_Samples': len(X_test),
        'Training_Time': training_time,
        'Inference_Time': inference_time * 1000,  # Convert to milliseconds
        'Train_Accuracy': train_acc,
        'Val_Accuracy': val_acc,
        'Test_Accuracy': test_acc,
        'Train_Loss': train_loss,
        'Val_Loss': val_loss,
        'Test_Loss': test_loss,
        'Learning_Rate': 1e-5 if fine_tune_layers > 0 else 1e-4,
        'Batch_Size': 32,
        'Epochs': best_epoch + 1,
        'Fine_Tuned_Layers': fine_tune_layers
    }
    results.append(model_results)

    print(f"{model_name} completed in {training_time:.1f}s | Test accuracy: {test_acc:.4f}")

# ======================
# 5. RESULTS ANALYSIS
# ======================
print("[5/6] Analyzing results...")
results_df = pd.DataFrame(results)

# Save all results
results_df.to_csv(f"{output_dir}/model_comparison.csv", index=False)

# Generate comparison plots
plt.figure(figsize=(12, 6))
sns.barplot(x="Model", y="Test_Accuracy", data=results_df)
plt.title("Model Accuracy Comparison")
plt.ylim(0, 1)
plt.xticks(rotation=45)
plt.savefig(f"{output_dir}/accuracy_comparison.png", bbox_inches='tight')
plt.close()

plt.figure(figsize=(12, 6))
sns.barplot(x="Model", y="Training_Time", data=results_df)
plt.title("Training Time Comparison (seconds)")
plt.xticks(rotation=45)
plt.savefig(f"{output_dir}/training_time_comparison.png", bbox_inches='tight')
plt.close()

# ======================
# 6. BEST MODEL DEPLOYMENT
# ======================
print("[6/6] Preparing best model for deployment...")
best_model_row = results_df.loc[results_df['Test_Accuracy'].idxmax()]
best_model_name = best_model_row['Model']

# Load best model
best_model = load_model(f"{output_dir}/{best_model_name}/model.h5")

# Save best model separately
best_model.save(f"{output_dir}/best_model.h5")

# # Create inference function example
# inference_code = f"""
# # Sample inference code for {best_model_name}
# from tensorflow.keras.preprocessing import image
# import numpy as np
# from tensorflow.keras.models import load_model

# def predict_bird_species(img_path):
#     img = image.load_img(img_path, target_size=(224, 224))
#     x = image.img_to_array(img)
#     x = np.expand_dims(x, axis=0) / 255.0

#     model = load_model("{output_dir}/best_model.h5")
#     preds = model.predict(x)
#     class_idx = np.argmax(preds[0])

#     with open("{output_dir}/class_names.txt") as f:
#         classes = [line.strip() for line in f.readlines()]

#     return classes[class_idx], float(preds[0][class_idx])

# # Example usage:
# # species, confidence = predict_bird_species("your_image.jpg")
# # print(f"Predicted: {species} (Confidence: {confidence:.2%})")
# """

# with open(f"{output_dir}/inference_example.py", "w") as f:
#     f.write(inference_code)

print(f"\nExperiment complete! All results saved to: {output_dir}")
print(f"Best model: {best_model_name} with accuracy {best_model_row['Test_Accuracy']:.4f}")

# Display results in a nice table
from IPython.display import display, HTML
display(HTML(results_df.to_html()))

[1/6] Loading and preprocessing data...
[2/6] Setting up data augmentation...

[4/6] Training VGG16...
[3/6] Building model with 5 fine-tuned layers...
58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step


/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/30
17/17 ━━━━━━━━━━━━━━━━━━━━ 34s 1s/step - accuracy: 0.2505 - loss: 1.9247 - val_accuracy: 0.2551 - val_loss: 1.7019 - learning_rate: 1.0000e-05
Epoch 2/30
 1/17 ━━━━━━━━━━━━━━━━━━━━ 2s 181ms/step - accuracy: 0.3125 - loss: 1.8086

/usr/local/lib/python3.11/dist-packages/keras/src/trainers/epoch_iterator.py:107: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 43ms/step - accuracy: 0.3125 - loss: 1.8086 - val_accuracy: 0.2551 - val_loss: 1.6950 - learning_rate: 1.0000e-05
Epoch 3/30
17/17 ━━━━━━━━━━━━━━━━━━━━ 7s 410ms/step - accuracy: 0.3714 - loss: 1.6174 - val_accuracy: 0.4592 - val_loss: 1.5612 - learning_rate: 1.0000e-05
Epoch 4/30
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 46ms/step - accuracy: 0.4688 - loss: 1.2559 - val_accuracy: 0.4796 - val_loss: 1.5537 - learning_rate: 1.0000e-05
Epoch 5/30
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 415ms/step - accuracy: 0.4489 - loss: 1.3490 - val_accuracy: 0.6939 - val_loss: 1.4257 - learning_rate: 1.0000e-05
Epoch 6/30
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 36ms/step - accuracy: 0.4375 - loss: 1.4687 - val_accuracy: 0.6939 - val_loss: 1.4184 - learning_rate: 1.0000e-05
Epoch 7/30
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 530ms/step - accuracy: 0.5759 - loss: 1.1392 - val_accuracy: 0.8163 - val_loss: 1.2794 - learning_rate: 1.0000e-05
Epoch 8/30
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 43ms/step - accuracy: 0.6562 - loss: 

...

6/6 ━━━━━━━━━━━━━━━━━━━━ 12s 1s/step


/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


EfficientNetB0 completed in 165.3s | Test accuracy: 0.1902
[5/6] Analyzing results...
[6/6] Preparing best model for deployment...



Experiment complete! All results saved to: /content/drive/MyDrive/SEM 6/ML/CODE/COMPARISON/bird_species_20250618_171140
Best model: MobileNetV2 with accuracy 0.9939


,Model,Trainable_Params,Non_Trainable_Params,Total_Params,Train_Samples,Val_Samples,Test_Samples,Training_Time,Inference_Time,Train_Accuracy,Val_Accuracy,Test_Accuracy,Train_Loss,Val_Loss,Test_Loss,Learning_Rate,Batch_Size,Epochs,Fine_Tuned_Layers
0,VGG16,7346182,7636288.0,14982470,550,98,163,164.028395,23.810141,0.913127,0.959184,0.981595,0.337464,0.304036,0.228760,0.00001,32,25,5
1,ResNet50,5518854,19123072.0,24641926,550,98,163,111.562639,32.769681,0.380515,0.265306,0.153374,1.618534,1.806946,1.861627,0.00001,32,11,10
2,MobileNetV2,659974,2259008.0,2918982,550,98,163,162.895319,32.829352,0.972973,0.989796,0.993865,0.102269,0.056215,0.042255,0.00010,32,25,0
3,Xception,7841574,14074120.0,21915694,550,98,163,185.068363,20.226411,0.940154,0.928571,0.975460,0.267365,0.251136,0.153512,0.00001,32,25,15
4,EfficientNetB0,1553206,3157363.0,4710569,550,98,163,165.271305,33.083181,0.152510,0.214286,0.190184,2.205363,1.772862,1.809346,0.00001,32,25,8
